In [24]:
from data_model_loader import *

model = load_model()
annotations, images = load_coco_2014_dataset()

with open('config.json', 'r') as file:
    config = json.load(file)

coco_folder = config['coco_folder']
annotation_file_path = config['annotation_file_path']

OD model already present locally.
data/coco2014\val2014.zip already exists. Skipping download.
data/coco2014\val2014 already exists. Skipping extraction.
data/coco2014\annotations_trainval2014.zip already exists. Skipping download.
data/coco2014\annotations_trainval2014 already exists. Skipping extraction.
COCO 2014 dataset download and extraction complete.
Get Annotations ..
Randomly selecting 1500 pictures ..
Done!


In [25]:
from PIL import Image
import torchvision.transforms as transforms
import pathlib
from tqdm import tqdm


def predict(model, data_path, n_images=5):
    data_path = pathlib.Path(data_path)
    predictions = {}

    for img in tqdm(images[:n_images], desc="Loading images"):
        # load image
        image_path = data_path/"val2014"/"val2014"/img
        image = Image.open(image_path)

        # transform to tensor
        transform = transforms.Compose([
            transforms.ToTensor()  # Konvertiert das Bild zu einem Tensor [C, H, W] mit Werten zwischen 0 und 1
        ])
        image_tensor = transform(image)
        image_tensor = (image_tensor * 255).byte()  # convert to uint8
        image_tensor = image_tensor.permute(1, 2, 0)  # [C, H, W] -> [H, W, C]
        image_tensor = image_tensor.unsqueeze(0)

        # inference
        detector_output = model(image_tensor)
        score = detector_output["detection_scores"].numpy()[0]
        detected_class = detector_output["detection_classes"].numpy()[0]
        
        # eval
        predictions[img] = {
            "detected_classes": detected_class,
            "scores": score
        }
        

    return predictions
    
def get_annotation(annotation_file_path):
    with open(annotation_file_path, 'r') as f:
        coco_data = json.load(f)
    category_map = {category['id']: category['name'] for category in coco_data['categories']}
    return category_map


In [26]:
results = predict(model, coco_folder)
results

Loading images: 100%|██████████| 5/5 [00:02<00:00,  2.49it/s]


{'COCO_val2014_000000480205.jpg': {'detected_classes': array([16., 16., 16., 34.,  2., 16., 16., 16., 34., 38., 37., 16., 16.,
         16., 16., 34., 34., 16., 16., 64., 50., 38., 16., 16.,  2.,  3.,
         15.,  2., 87., 17.,  9., 37.,  4.,  1., 16., 16., 34., 16., 16.,
         16., 48., 64., 16., 18., 38., 16., 42., 20., 35., 11., 16., 16.,
          2., 23., 16., 16., 16., 16., 31.,  2., 38., 16., 16.,  2., 35.,
         38.,  2., 16., 35., 38., 27.,  7., 25., 16., 52.,  1., 24., 24.,
         34., 16., 18., 49., 42.,  2., 64., 16., 16.,  9., 24., 15., 64.,
         28., 49., 16., 33., 53., 64., 38., 16., 16.], dtype=float32),
  'scores': array([0.866809  , 0.43555936, 0.3826751 , 0.35660893, 0.34681007,
         0.26023078, 0.23864384, 0.23520882, 0.23046525, 0.22098823,
         0.22060794, 0.19058824, 0.18708159, 0.1819129 , 0.1815712 ,
         0.169781  , 0.16579697, 0.16056225, 0.15562406, 0.14905538,
         0.14285602, 0.14026168, 0.1360578 , 0.13487563, 0.13349326,
   

In [27]:
classes = get_annotation(annotation_file_path)

In [28]:
classes

{1: 'person',
 2: 'bicycle',
 3: 'car',
 4: 'motorcycle',
 5: 'airplane',
 6: 'bus',
 7: 'train',
 8: 'truck',
 9: 'boat',
 10: 'traffic light',
 11: 'fire hydrant',
 13: 'stop sign',
 14: 'parking meter',
 15: 'bench',
 16: 'bird',
 17: 'cat',
 18: 'dog',
 19: 'horse',
 20: 'sheep',
 21: 'cow',
 22: 'elephant',
 23: 'bear',
 24: 'zebra',
 25: 'giraffe',
 27: 'backpack',
 28: 'umbrella',
 31: 'handbag',
 32: 'tie',
 33: 'suitcase',
 34: 'frisbee',
 35: 'skis',
 36: 'snowboard',
 37: 'sports ball',
 38: 'kite',
 39: 'baseball bat',
 40: 'baseball glove',
 41: 'skateboard',
 42: 'surfboard',
 43: 'tennis racket',
 44: 'bottle',
 46: 'wine glass',
 47: 'cup',
 48: 'fork',
 49: 'knife',
 50: 'spoon',
 51: 'bowl',
 52: 'banana',
 53: 'apple',
 54: 'sandwich',
 55: 'orange',
 56: 'broccoli',
 57: 'carrot',
 58: 'hot dog',
 59: 'pizza',
 60: 'donut',
 61: 'cake',
 62: 'chair',
 63: 'couch',
 64: 'potted plant',
 65: 'bed',
 67: 'dining table',
 70: 'toilet',
 72: 'tv',
 73: 'laptop',
 74: 'mo